In [1]:
import pandas as pd
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

C:\Users\YZLT0221\AppData\Local\Temp\1\ipykernel_15452\2456827457.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
import pandas as pd

def filter_data_for_month_year(df, month, year):
    return df[(df['ladate'].dt.year == year) & (df['ladate'].dt.month == month)]

def filter_data_based_on_columns(df, e_offer_values, c_data_type_values, country_list):
    return df[
        df['E_OFFER'].isin(e_offer_values) &
        df['C_DATA_TYPE'].isin(c_data_type_values) &
        df['country'].isin(country_list)
    ]

def calculate_sums_and_shares(df, columns_to_sum, month, year):
    sum_by_country = df.groupby('country')[columns_to_sum].sum().reset_index()
    sum_by_country['E2E_Digital_share'] = ((sum_by_country['E2E_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['Assisted_Digital_share'] = ((sum_by_country['Assisted_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['share_digital_all'] = (((sum_by_country['E2E_Digital_sales'] + sum_by_country['Assisted_Digital_sales']) / sum_by_country['All_channels_sales']) * 100).round(1)

    # Assign month and year to the dataframe
    sum_by_country['month'] = month
    sum_by_country['year'] = year
    return sum_by_country

def add_total_row(sum_by_country, columns_to_sum, month, year):
    # Sum the specified columns across all countries
    total_sum = sum_by_country[columns_to_sum].sum()
    # Calculate the total E2E Digital share percentage
    total_e2e_digital_share = (total_sum['E2E_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Assisted Digital share percentage
    total_assisted_digital_share = (total_sum['Assisted_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Digital share percentage
    total_share_digital_all = ((total_sum['E2E_Digital_sales'] + total_sum['Assisted_Digital_sales']) / total_sum['All_channels_sales']) * 100

    # Create a dataframe for the total row
    total_row = pd.DataFrame(data={
        'E2E_Digital_sales': [total_sum['E2E_Digital_sales']],
        'Assisted_Digital_sales': [total_sum['Assisted_Digital_sales']],
        'All_channels_sales': [total_sum['All_channels_sales']],
        'E2E_Digital_share': [round(total_e2e_digital_share, 1)],
        'Assisted_Digital_share': [round(total_assisted_digital_share, 1)],
        'share_digital_all': [round(total_share_digital_all, 1)],
        'month': [month],
        'year': [year],
        'country': 'Total'
    })

    # Concatenate the total row with the original dataframe
    sum_by_country_with_total = pd.concat([sum_by_country, total_row])

    return sum_by_country_with_total

def process_data(df, e_offer_values, c_data_type_values, columns_to_sum, country_list):
    results = []

    # Get unique months and years from the dataframe
    unique_months_years = df[['ladate']].apply(lambda x: (x['ladate'].month, x['ladate'].year), axis=1).unique()

    for month, year in unique_months_years:
        # Filter data for the current month and year
        filtered_df = filter_data_for_month_year(df, month, year)
        filtered_df = filter_data_based_on_columns(filtered_df, e_offer_values, c_data_type_values, country_list)
        sum_by_country = calculate_sums_and_shares(filtered_df, columns_to_sum, month, year)
        sum_by_country_with_total = add_total_row(sum_by_country, columns_to_sum, month, year)
        results.append(sum_by_country_with_total)
        
    # Concatenate all results into a single dataframe
    final_result = pd.concat(results)

    return final_result


In [2]:
# Define the filter values
e_offer_values = ['Mobile Only postpaid', 'Fixed Only', 'Mobile Convergent postpaid', 'Fixed Convergent']
c_data_type_values = ['Acquisitions', 'Renewals']
columns_to_sum = ['E2E_Digital_sales', 'Assisted_Digital_sales', 'All_channels_sales']
countries = ['opl', 'obe', 'oro', 'osk', 'omd', 'olu']

In [3]:
# Load the CSV file
file_path = 'Aug raw data.csv'  # Replace with the actual file path
df = pd.read_csv(file_path)



In [4]:
df.head()

,ladate,country,C_DATA_TYPE,D_BRAND,E_OFFER,Digital_sales,E2E_Digital_sales,Assisted_Digital_sales,Pick_up_in_store,All_channels_sales,Digital_share,E2E_Digital_share,Assisted_Digital_share
0,11/1/2020,osp,TV contracts acquisitions,Orange,TV,4926.0,501.0,4425.0,NaN,17944.0,0.274521,0.027920,0.246601
1,10/1/2021,osp,TV contracts acquisitions,Orange,TV,3658.0,889.0,2769.0,NaN,16122.0,0.226895,0.055142,0.171753
2,9/1/2021,osp,TV contracts acquisitions,Orange,TV,4964.0,1080.0,3884.0,NaN,20807.0,0.238574,0.051906,0.186668
3,12/1/2021,osp,TV contracts acquisitions,Orange,TV,2800.0,765.0,2035.0,NaN,11891.0,0.235472,0.064334,0.171138
4,6/1/2020,osp,TV contracts acquisitions,Orange,TV,3421.0,451.0,2970.0,NaN,16020.0,0.213546,0.028152,0.185393


In [5]:
df['ladate'] = pd.to_datetime(df['ladate'])

In [10]:
result = process_data(df, e_offer_values, c_data_type_values, columns_to_sum, countries)

In [12]:
result[(result['month'] == 8) & (result['year'] == 2024)]

,country,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all,month,year
0,obe,10976.0,4432.0,78036.0,14.1,5.7,19.7,8,2024
1,olu,221.0,58.0,2345.0,9.4,2.5,11.9,8,2024
2,omd,0.0,1275.0,24243.0,0.0,5.3,5.3,8,2024
3,opl,52555.0,17632.0,238160.0,22.1,7.4,29.5,8,2024
4,oro,25275.0,1191.0,163833.0,15.4,0.7,16.2,8,2024
5,osk,1782.0,1701.0,31597.0,5.6,5.4,11.0,8,2024
0,Total,90809.0,26289.0,538214.0,16.9,4.9,21.8,8,2024


In [6]:
import pandas as pd

def filter_data_for_month_year(df, month, year):
    return df[(df['ladate'].dt.year == year) & (df['ladate'].dt.month == month)]

def filter_data_based_on_columns(df, e_offer_values, c_data_type_values, country_list):
    return df[
        df['E_OFFER'].isin(e_offer_values) &
        df['C_DATA_TYPE'].isin(c_data_type_values) &
        df['country'].isin(country_list)
    ]

def calculate_sums_and_shares(df, columns_to_sum, month, year):
    sum_by_country = df.groupby('country')[columns_to_sum].sum().reset_index()
    sum_by_country['E2E_Digital_share'] = ((sum_by_country['E2E_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['Assisted_Digital_share'] = ((sum_by_country['Assisted_Digital_sales'] / sum_by_country['All_channels_sales']) * 100).round(1)
    sum_by_country['share_digital_all'] = (((sum_by_country['E2E_Digital_sales'] + sum_by_country['Assisted_Digital_sales']) / sum_by_country['All_channels_sales']) * 100).round(1)

    # Assign month and year to the dataframe
    sum_by_country['month'] = month
    sum_by_country['year'] = year
    return sum_by_country

def add_total_row(sum_by_country, columns_to_sum, month, year):
    # Sum the specified columns across all countries
    total_sum = sum_by_country[columns_to_sum].sum()
    # Calculate the total E2E Digital share percentage
    total_e2e_digital_share = (total_sum['E2E_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Assisted Digital share percentage
    total_assisted_digital_share = (total_sum['Assisted_Digital_sales'] / total_sum['All_channels_sales']) * 100
    # Calculate the total Digital share percentage
    total_share_digital_all = ((total_sum['E2E_Digital_sales'] + total_sum['Assisted_Digital_sales']) / total_sum['All_channels_sales']) * 100

    # Create a dataframe for the total row
    total_row = pd.DataFrame(data={
        'E2E_Digital_sales': [total_sum['E2E_Digital_sales']],
        'Assisted_Digital_sales': [total_sum['Assisted_Digital_sales']],
        'All_channels_sales': [total_sum['All_channels_sales']],
        'E2E_Digital_share': [round(total_e2e_digital_share, 1)],
        'Assisted_Digital_share': [round(total_assisted_digital_share, 1)],
        'share_digital_all': [round(total_share_digital_all, 1)],
        'month': [month],
        'year': [year],
        'country': 'Total'
    })

    # Concatenate the total row with the original dataframe
    sum_by_country_with_total = pd.concat([sum_by_country, total_row])

    return sum_by_country_with_total

def calculate_ytd(final_result):
    # Calculate cumulative sums for YTD, ensuring it's done per year
    final_result['YTD_E2E_Digital_sales'] = final_result.groupby(['country', 'year'])['E2E_Digital_sales'].cumsum()
    final_result['YTD_Assisted_Digital_sales'] = final_result.groupby(['country', 'year'])['Assisted_Digital_sales'].cumsum()
    final_result['YTD_All_channels_sales'] = final_result.groupby(['country', 'year'])['All_channels_sales'].cumsum()
    
    # YTD shares can be calculated based on the cumulative sums
    final_result['E2E_Digital_share_YTD'] = ((final_result['YTD_E2E_Digital_sales'] / final_result['YTD_All_channels_sales']) * 100).round(1)
    final_result['Assisted_Digital_share_YTD'] = ((final_result['YTD_Assisted_Digital_sales'] / final_result['YTD_All_channels_sales']) * 100).round(1)
    final_result['share_digital_all_YTD'] = ((final_result['YTD_E2E_Digital_sales'] + final_result['YTD_Assisted_Digital_sales']) / final_result['YTD_All_channels_sales'] * 100).round(1)
    
    return final_result

def process_data(df, e_offer_values, c_data_type_values, columns_to_sum, country_list):
    results = []

    # Get unique months and years from the dataframe
    unique_months_years = df[['ladate']].apply(lambda x: (x['ladate'].month, x['ladate'].year), axis=1).unique()

    for month, year in unique_months_years:
        # Filter data for the current month and year
        filtered_df = filter_data_for_month_year(df, month, year)
        filtered_df = filter_data_based_on_columns(filtered_df, e_offer_values, c_data_type_values, country_list)
        sum_by_country = calculate_sums_and_shares(filtered_df, columns_to_sum, month, year)
        sum_by_country_with_total = add_total_row(sum_by_country, columns_to_sum, month, year)
        results.append(sum_by_country_with_total)
        
    # Concatenate all results into a single dataframe
    final_result = pd.concat(results)

    # Calculate YTD
    final_result = calculate_ytd(final_result)

    return final_result


In [7]:
result = process_data(df, e_offer_values, c_data_type_values, columns_to_sum, countries)
result[(result['month'] == 8) & (result['year'] == 2024)]

,country,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all,month,year,YTD_E2E_Digital_sales,YTD_Assisted_Digital_sales,YTD_All_channels_sales,E2E_Digital_share_YTD,Assisted_Digital_share_YTD,share_digital_all_YTD
0,obe,10976.0,4432.0,78036.0,14.1,5.7,19.7,8,2024,88876.0,33629.0,620311.0,14.3,5.4,19.7
1,olu,221.0,58.0,2345.0,9.4,2.5,11.9,8,2024,1349.0,183.0,19780.0,6.8,0.9,7.7
2,omd,0.0,1275.0,24243.0,0.0,5.3,5.3,8,2024,0.0,9171.0,163984.0,0.0,5.6,5.6
3,opl,52555.0,17632.0,238160.0,22.1,7.4,29.5,8,2024,398413.0,121955.0,1907183.0,20.9,6.4,27.3
4,oro,25275.0,1191.0,163833.0,15.4,0.7,16.2,8,2024,176511.0,9733.0,1413112.0,12.5,0.7,13.2
5,osk,1782.0,1701.0,31597.0,5.6,5.4,11.0,8,2024,19913.0,13721.0,287513.0,6.9,4.8,11.7
0,Total,90809.0,26289.0,538214.0,16.9,4.9,21.8,8,2024,685062.0,188392.0,4411883.0,15.5,4.3,19.8


In [13]:
result[(result['month'] == 8) & (result['year'] == 2023)]

,country,E2E_Digital_sales,Assisted_Digital_sales,All_channels_sales,E2E_Digital_share,Assisted_Digital_share,share_digital_all,month,year,YTD_E2E_Digital_sales,YTD_Assisted_Digital_sales,YTD_All_channels_sales,E2E_Digital_share_YTD,Assisted_Digital_share_YTD,share_digital_all_YTD,date
0,obe,6385.0,4081.0,58311.0,10.9,7.0,17.9,8,2023,6385.0,4081.0,58311.0,10.9,7.0,17.9,2023-08-01
1,olu,236.0,0.0,2254.0,10.5,0.0,10.5,8,2023,236.0,0.0,2254.0,10.5,0.0,10.5,2023-08-01
2,omd,0.0,1200.0,23432.0,0.0,5.1,5.1,8,2023,0.0,1200.0,23432.0,0.0,5.1,5.1,2023-08-01
3,opl,46757.0,12179.0,238628.0,19.6,5.1,24.7,8,2023,46757.0,12179.0,238628.0,19.6,5.1,24.7,2023-08-01
4,oro,19295.0,1468.0,190759.0,10.1,0.8,10.9,8,2023,19295.0,1468.0,190759.0,10.1,0.8,10.9,2023-08-01
5,osk,3406.0,1686.0,38456.0,8.9,4.4,13.2,8,2023,3406.0,1686.0,38456.0,8.9,4.4,13.2,2023-08-01
0,Total,76079.0,20614.0,551840.0,13.8,3.7,17.5,8,2023,76079.0,20614.0,551840.0,13.8,3.7,17.5,2023-08-01


In [8]:
def reorder_columns(df):
    # Specify the desired column order, with 'month' and 'year' as the first columns
    cols = ['month', 'year', 'country', 'E2E_Digital_sales', 'Assisted_Digital_sales', 'All_channels_sales', 
            'E2E_Digital_share', 'Assisted_Digital_share', 'share_digital_all', 
            'E2E_Digital_sales_YTD', 'Assisted_Digital_sales_YTD', 'All_channels_sales_YTD', 
            'E2E_Digital_share_YTD', 'Assisted_Digital_share_YTD', 'share_digital_all_YTD']
    
    # Reorder the columns in the dataframe
    df = df[cols]
    
    return df

In [11]:

output=[]
df=result
df['date'] = pd.to_datetime(df[['year', 'month']].assign(day=1))
df = df.sort_values(by='date', ascending=[False])
share_columns = ['E2E_Digital_share', 'Assisted_Digital_share', 'share_digital_all','E2E_Digital_share_YTD', 'Assisted_Digital_share_YTD', 'share_digital_all_YTD']
#share_columns = ['E2E_Digital_share', 'Assisted_Digital_share', 'share_digital_all', 'E2E_Digital_share_YTD', 'Assisted_Digital_share_YTD', 'share_digital_all_YTD']

for idx, row in df.iterrows():
    for share_col in share_columns:
        current_value = row[share_col]
        previous_year = row['year'] - 1
        previous_value_row = df[(df['month'] == row['month']) & (df['year'] == previous_year) & (df['country'] == row['country'])]
        country = row['country'].upper()
        if not previous_value_row.empty:
            previous_value = previous_value_row[share_col].iloc[0]
            difference = current_value - previous_value  # Calculate the difference
            output.append((row['month'], row['year'], country, share_col, current_value, previous_value,difference))

output_df = pd.DataFrame(output, columns=['month', 'year', 'country', 'share_column', 'current_month_value', 'YoY_month_value','difference'])

In [12]:
output_df[(output_df['month'] == 8) & (output_df['year'] == 2024) & (output_df['country'] == 'TOTAL')]

,month,year,country,share_column,current_month_value,YoY_month_value,difference
36,8,2024,TOTAL,E2E_Digital_share,16.9,13.8,3.1
37,8,2024,TOTAL,Assisted_Digital_share,4.9,3.7,1.2
38,8,2024,TOTAL,share_digital_all,21.8,17.5,4.3
39,8,2024,TOTAL,E2E_Digital_share_YTD,15.5,13.8,1.7
40,8,2024,TOTAL,Assisted_Digital_share_YTD,4.3,3.7,0.6
41,8,2024,TOTAL,share_digital_all_YTD,19.8,17.5,2.3


In [35]:
output_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1014 entries, 0 to 1013
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   month                1014 non-null   int64  
 1   year                 1014 non-null   int64  
 2   country              1014 non-null   object 
 3   share_column         1014 non-null   object 
 4   current_month_value  1014 non-null   float64
 5   YoY_month_value      1002 non-null   float64
 6   difference           1002 non-null   float64
dtypes: float64(3), int64(2), object(2)
memory usage: 55.6+ KB
